# Séance 6 — Agrégation, jointures et modèle en étoile

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 6.1 à 6.4

## Ce que vous saurez faire à la fin

- distinguer `agg`, `transform` et `apply` et choisir le bon ;
- contrôler la cardinalité d'une jointure et diagnostiquer une explosion de lignes ;
- passer du format large au format long et inversement ;
- livrer un modèle en étoile exploitable directement par Power BI ou Tableau.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [12]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  .gitkeep
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


In [13]:
sales = load_dataset('ventes_transformees')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
products = pd.read_csv(RAW_DIR / 'produits.csv')
stores = pd.read_csv(RAW_DIR / 'magasins.csv')

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')

for label, frame in [('sales', sales), ('customers', customers),
                     ('products', products), ('stores', stores)]:
    print(f'{label:<10} {frame.shape}')

sales      (16992, 23)
customers  (3060, 5)
products   (39, 5)
stores     (9, 5)


---
## Atelier 6.1 — groupby : agg, transform, apply (50 min)

### Partie guidée

In [14]:
# agg : réduit chaque groupe à une ligne
by_category = sales.groupby('categorie').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    panier_median=('montant', 'median'),
    quantite_totale=('quantite', 'sum'),
).round(2).sort_values('ca_total', ascending=False)

by_category

,nb_commandes,ca_total,panier_moyen,panier_median,quantite_totale
categorie,,,,,
Informatique,5162,17445914.35,3379.68,1167.38,26602
Audio,3949,9234318.11,2338.39,1073.00,15078
Mobilier,3936,8363145.49,2124.78,717.48,18274
Accessoire,3870,7575740.71,1957.56,834.65,14978


Comparez `panier_moyen` et `panier_median`. Un écart important signale une distribution
asymétrique : quelques commandes très élevées tirent la moyenne vers le haut. Publier la
seule moyenne dans un tableau de bord donnerait une image fausse du panier typique.

In [15]:
# transform : renvoie un résultat ALIGNÉ sur les lignes d'origine
sales['ca_categorie'] = sales.groupby('categorie')['montant'].transform('sum')
sales['part_dans_categorie'] = (sales['montant'] / sales['ca_categorie'] * 100).round(4)

print(sales[['categorie', 'montant', 'ca_categorie', 'part_dans_categorie']].head())
print('\nSomme des parts par catégorie :')
print(sales.groupby('categorie')['part_dans_categorie'].sum().round(2))

      categorie  montant  ca_categorie  part_dans_categorie
0  Informatique   457.88   17445914.35               0.0026
1      Mobilier   140.96    8363145.49               0.0017
2         Audio  2073.36    9234318.11               0.0225
3  Informatique  2281.00   17445914.35               0.0131
4      Mobilier  1904.49    8363145.49               0.0228

Somme des parts par catégorie :
categorie
Accessoire      100.0
Audio           100.0
Informatique    100.0
Mobilier        100.0
Name: part_dans_categorie, dtype: float64


**La règle est simple.** `agg` réduit (une ligne par groupe), `transform` conserve
(autant de lignes qu'à l'entrée). Chaque fois que vous voulez comparer une valeur à la
statistique de son groupe, c'est `transform` qu'il vous faut, jamais un `merge` sur un
résultat de `agg`.

In [16]:
# apply : le plus souple, mais le plus lent. À réserver aux cas irréductibles.
def summarize_group(group):
    return pd.Series({
        'nb': len(group),
        'top_produit': group['id_produit'].mode().iloc[0] if len(group) else None,
        'part_annulees': (group['statut'] == 'annule').mean().round(3),
    })


sales.groupby('canal').apply(summarize_group, include_groups=False)

,nb,top_produit,part_annulees
canal,,,
boutique,5738,P0027,0.165
telephone,2865,P0015,0.166
web,8389,P0038,0.170


### Partie autonome

In [17]:
# Q1. CA par catégorie et canal, avec marges.
ca_categorie_canal = sales.pivot_table(
    index='categorie',
    columns='canal',
    values='montant',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
    fill_value=0,
).round(2)

ca_categorie_canal

canal,boutique,telephone,web,Total
categorie,,,,
Accessoire,2168517.47,2311009.90,3096213.34,7575740.71
Audio,2826937.75,861985.02,5545395.34,9234318.11
Informatique,6269923.49,4832327.05,6343663.81,17445914.35
Mobilier,3379259.98,2469037.09,2514848.42,8363145.49
Total,14644638.69,10474359.06,17500120.91,42619118.66


In [18]:
# Q2. Une ligne par client, puis les dix clients au CA le plus élevé.
customer_summary = sales.groupby('id_client').agg(
    nb_commandes=('id_commande', 'nunique'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    premiere_commande=('date_commande', 'min'),
    derniere_commande=('date_commande', 'max'),
    nb_categories=('categorie', 'nunique'),
).sort_values('ca_total', ascending=False)

top_10_clients = customer_summary.head(10).round({'ca_total': 2, 'panier_moyen': 2})
top_10_clients

,nb_commandes,ca_total,panier_moyen,premiere_commande,derniere_commande,nb_categories
id_client,,,,,,
C00696,7,1110180.18,158597.17,2023-03-13 00:00:00,2024-11-19 00:00:00,3
C02327,4,907834.48,226958.62,2023-10-31 00:00:00,2024-12-28 14:36:00,2
C00279,10,876999.45,87699.94,2023-01-24 00:00:00,2024-11-05 00:00:00,4
C02631,10,788266.88,78826.69,2023-01-12 00:00:00,2024-12-19 00:00:00,4
C00941,7,783179.31,111882.76,2023-01-29 00:00:00,2024-09-15 00:00:00,3
C00355,13,779167.48,59935.96,2023-04-09 16:29:00,2024-09-25 00:00:00,4
C02101,6,750743.89,125123.98,2023-01-05 11:00:00,2024-02-03 00:00:00,3
C00806,5,707386.06,141477.21,2023-05-03 00:00:00,2024-08-26 00:00:00,2
C02285,7,687164.50,98166.36,2023-01-10 10:43:00,2024-12-27 00:00:00,2


In [19]:
# Q3. Le seuil propre à chaque catégorie est calculé avec transform :
# le résultat reste aligné sur chaque commande.
sales['mediane_categorie'] = sales.groupby('categorie')['montant'].transform('median')
sales['commande_au_dessus_seuil'] = sales['montant'] > 1.5 * sales['mediane_categorie']

part_commandes_au_dessus = sales['commande_au_dessus_seuil'].mean()
print(f"Commandes au-dessus de 1,5 × la médiane de leur catégorie : "
      f"{sales['commande_au_dessus_seuil'].sum():,} / {len(sales):,} "
      f"({part_commandes_au_dessus:.2%})")

Commandes au-dessus de 1,5 × la médiane de leur catégorie : 5,662 / 16,992 (33.32%)


---
## Atelier 6.2 — Jointures et explosion de lignes (55 min)

### Partie guidée

In [20]:
print('Lignes avant jointure :', len(sales))

with_products = sales.merge(products, on='id_produit', how='left', suffixes=('', '_prod'))
print('Après jointure produits :', len(with_products))

n_orphans = with_products['libelle'].isna().sum()
print(f'Lignes sans produit correspondant : {n_orphans}',
      f'({n_orphans / len(with_products):.2%})')

unknown_ids = sales.loc[~sales['id_produit'].isin(products['id_produit']), 'id_produit']
print('Identifiants fautifs :', unknown_ids.unique())

Lignes avant jointure : 16992
Après jointure produits : 16992
Lignes sans produit correspondant : 75 (0.44%)
Identifiants fautifs : <ArrowStringArray>
['P9999']
Length: 1, dtype: str


**Le réflexe à acquérir : compter les lignes avant et après chaque jointure.** Un `left join`
conserve le nombre de lignes *seulement si* la clé est unique à droite. Si elle ne l'est pas,
le résultat se multiplie sans avertissement.

In [21]:
# La table clients est saine : sa clé primaire est bien unique.
print('Identifiants clients dupliqués :', customers['id_client'].duplicated().sum())

# On simule donc le cas d'une table de droite dégradée — situation banale dès qu'une
# extraction est relancée deux fois ou qu'un export concatène deux périodes.
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
exploded = sales.merge(customers_with_duplicates, on='id_client', how='left')
print(f'{len(sales)} lignes -> {len(exploded)} après jointure sur une clé non unique')
print('Aucune erreur levée. Le problème ne se verra que bien plus loin.')

Identifiants clients dupliqués : 0
16992 lignes -> 17287 après jointure sur une clé non unique
Aucune erreur levée. Le problème ne se verra que bien plus loin.


In [22]:
# Le paramètre validate transforme le problème silencieux en erreur explicite
try:
    sales.merge(customers_with_duplicates, on='id_client', how='left', validate='many_to_one')
except Exception as error:
    print(type(error).__name__, ':', error)

MergeError : Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 id_client
   C00001
   C00002
   C00003
   C00004
   C00005 ...


Prenez l'habitude de passer `validate` sur **toutes** vos jointures. Le coût est nul et
l'erreur qu'il attrape est parmi les plus coûteuses à diagnostiquer plus tard.

### Partie autonome

In [23]:
# Q4. On conserve les commandes orphelines dans le left join : elles restent
# analysables et les attributs de dimension manquants sont explicitement repérés.
import sys
sys.path.insert(0, str(ROOT / 'src'))
from relational import safe_merge

print('Lignes de ventes au départ :', len(sales))
sales_for_analysis = sales.drop(
    columns=[column for column in products.columns
             if column != 'id_produit' and column in sales.columns]
)
analysis = safe_merge(
    sales_for_analysis,
    products.drop_duplicates('id_produit'),
    on='id_produit',
    how='left',
    validate='many_to_one',
    suffixes=('', '_produit'),
)
print('Après produits :', len(analysis))
orphan_product_ids = analysis.loc[analysis['libelle'].isna(), 'id_produit'].unique()
print('Produit(s) sans correspondance :', orphan_product_ids.tolist(),
      '— lignes conservées avec attributs produit manquants.')

analysis = safe_merge(
    analysis, customers.drop_duplicates('id_client'),
    on='id_client', how='left', validate='many_to_one',
    suffixes=('', '_client'),
)
print('Après clients :', len(analysis))
orphan_customer_ids = analysis.loc[analysis['segment'].isna(), 'id_client'].unique()
print('Clients sans correspondance :', orphan_customer_ids.tolist())

analysis = safe_merge(
    analysis, stores.drop_duplicates('id_magasin'),
    on='id_magasin', how='left', validate='many_to_one',
    suffixes=('', '_magasin'),
)
print('Après magasins :', len(analysis))
orphan_store_ids = analysis.loc[analysis['nom_magasin'].isna(), 'id_magasin'].unique()
print('Magasins sans correspondance :', orphan_store_ids.tolist())

assert len(analysis) == len(sales), 'Une jointure a modifié le nombre de lignes'
analysis.head()

Lignes de ventes au départ : 16992
safe_merge : 75 ligne(s) non appariée(s) sur 16992 (0.44% des lignes de gauche)
Après produits : 16992
Produit(s) sans correspondance : ['P9999'] — lignes conservées avec attributs produit manquants.
safe_merge : 0 ligne(s) non appariée(s) sur 16992 (0.00% des lignes de gauche)
Après clients : 16992
Clients sans correspondance : []
safe_merge : 0 ligne(s) non appariée(s) sur 16992 (0.00% des lignes de gauche)
Après magasins : 16992
Magasins sans correspondance : []


,id_commande,date_commande,id_client,id_produit,id_magasin,quantite,prix_unitaire,remise_pct,canal,statut,ville_livraison,quantite_suspecte,montant,marge_unitaire,taux_marge,commande_non_aboutie,panier_moyen_client,ecart_prix_median_categorie,marge_commande,ca_categorie,part_dans_categorie,mediane_categorie,commande_au_dessus_seuil,libelle,categorie,sous_categorie,cout_achat,date_inscription,segment,ville,age,nom_magasin,ville_magasin,region,surface_m2
0,CMD0003112,2024-01-18 19:53:00,C02300,P0009,M05,3,160.66,5.0,telephone,retourne,lille,False,457.88,74.25,0.4622,1,1478.87,-0.8138,198.65,17445914.35,0.0026,1167.385,False,Clavier modele 3,Informatique,Clavier,86.41,2021-04-18,Particulier,PARIS,39.0,Boutique Bordeaux,Bordeaux,Nouvelle-Aquitaine,769
1,CMD0000107,2024-01-08 00:00:00,C02791,P0028,M03,2,70.48,0.0,boutique,livre,toulouse,False,140.96,35.67,0.5061,0,860.97,-0.8445,71.34,8363145.49,0.0017,717.480,False,Lampe modele 1,Mobilier,Lampe,34.81,2024-02-19,Particulier,PARIS,36.0,Boutique Marseille,Marseille,PACA,460
2,CMD0010050,2023-04-15 00:00:00,990031,P0017,M04,3,691.12,0.0,boutique,livre,marseille,False,2073.36,343.71,0.4973,0,1170.66,-0.0518,1031.13,9234318.11,0.0225,1073.000,True,Enceinte modele 2,Audio,Enceinte,347.41,2019-02-24,Professionnel,Bordeaux,NaN,Boutique Lille,Lille,Hauts-de-France,245
3,CMD0001979,2024-11-06 00:00:00,C01883,P0010,M04,4,570.25,0.0,web,retourne,marseille,False,2281.00,286.61,0.5026,1,1225.82,-0.3391,1146.44,17445914.35,0.0131,1167.385,True,Souris modele 1,Informatique,Souris,283.64,2019-06-01,Particulier,Nantes,77.0,Boutique Lille,Lille,Hauts-de-France,245
4,CMD0010513,2024-11-02 00:00:00,C00571,P0024,M02,2,1058.05,10.0,web,livre,marseille,False,1904.49,503.46,0.4758,0,1392.78,1.3345,795.31,8363145.49,0.0228,717.480,True,Bureau modele 3,Mobilier,Bureau,554.59,2021-01-02,Particulier,Strasbourg,44.0,Boutique Lyon,Lyon,Auvergne-Rhone-Alpes,853


In [24]:
# Q5. L'implémentation réutilisable est dans src/relational.py.
# Elle avertit du taux de lignes non appariées et protège la cardinalité d'un left join.
from relational import safe_merge

print('safe_merge importée depuis src/relational.py ; utilisée aux trois étapes de Q4.')

safe_merge importée depuis src/relational.py ; utilisée aux trois étapes de Q4.


---
## Atelier 6.3 — Formats long et large (35 min)

In [25]:
# Format large : une colonne par mois, lisible par un humain
wide = sales.pivot_table(index='categorie',
                         columns=sales['date_commande'].dt.month,
                         values='montant', aggfunc='sum').round(0)
wide.columns = [f'mois_{month:02d}' for month in wide.columns]
wide.head()

,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,367032.0,296781.0,333273.0,1218847.0,539612.0,339807.0,334525.0,1204402.0,491953.0,1090514.0,893844.0,465151.0
Audio,1180732.0,658118.0,389644.0,1082484.0,427252.0,1169529.0,575026.0,1057970.0,418011.0,431277.0,1169598.0,674677.0
Informatique,1808515.0,1586185.0,2721960.0,676080.0,2323577.0,750591.0,2923108.0,1046418.0,565554.0,596944.0,773439.0,1673544.0
Mobilier,849867.0,328436.0,369598.0,1133552.0,1356966.0,873502.0,361750.0,195968.0,1405982.0,300714.0,715121.0,471687.0


In [26]:
# Format long : une ligne par observation, exploitable par une machine
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')
long.head()

,categorie,mois,ca
0,Accessoire,mois_01,367032.0
1,Audio,mois_01,1180732.0
2,Informatique,mois_01,1808515.0
3,Mobilier,mois_01,849867.0
4,Accessoire,mois_02,296781.0


**Lequel choisir ?** Le format long pour tout traitement programmatique et pour les outils
de BI, qui attendent une ligne par fait. Le format large uniquement pour l'affichage final
destiné à un lecteur humain.

In [27]:
# Q6. set_index + unstack reconstruit le tableau sans pivot_table.
wide_from_long = (
    long.set_index(['categorie', 'mois'])['ca']
        .unstack('mois')
        .reindex(index=wide.index, columns=wide.columns)
)
pd.testing.assert_frame_equal(wide_from_long, wide, check_exact=True)
print(f"Reconstruction exacte : {wide_from_long.shape}, "
      f"{len(wide_from_long.columns)} colonnes mensuelles.")
wide_from_long.head()

Reconstruction exacte : (4, 12), 12 colonnes mensuelles.


,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,367032.0,296781.0,333273.0,1218847.0,539612.0,339807.0,334525.0,1204402.0,491953.0,1090514.0,893844.0,465151.0
Audio,1180732.0,658118.0,389644.0,1082484.0,427252.0,1169529.0,575026.0,1057970.0,418011.0,431277.0,1169598.0,674677.0
Informatique,1808515.0,1586185.0,2721960.0,676080.0,2323577.0,750591.0,2923108.0,1046418.0,565554.0,596944.0,773439.0,1673544.0
Mobilier,849867.0,328436.0,369598.0,1133552.0,1356966.0,873502.0,361750.0,195968.0,1405982.0,300714.0,715121.0,471687.0


---
## Atelier 6.4 — Livrer un modèle en étoile (40 min)

Un outil de BI attend une **table de faits** (une ligne par événement mesurable, contenant
des clés étrangères et des mesures numériques) entourée de **tables de dimensions** (une ligne
par entité, contenant ses attributs descriptifs). Livrer une seule table plate oblige
l'utilisateur à refaire ce travail dans l'outil, souvent moins bien.

In [28]:
# Dimension temps : indispensable, souvent oubliée
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')

dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5

print(dim_date.shape)
dim_date.head(3)

(730, 10)


,date,cle_date,annee,mois,nom_mois,trimestre,semaine_iso,jour_semaine,nom_jour,est_weekend
0,2023-01-01,20230101,2023,1,January,1,52,6,Sunday,True
1,2023-01-02,20230102,2023,1,January,1,1,0,Monday,False
2,2023-01-03,20230103,2023,1,January,1,1,1,Tuesday,False


La dimension temps couvre **toute la plage en continu**, y compris les jours sans aucune
vente. Sans elle, un graphique d'évolution saute les jours vides et donne une impression
de continuité trompeuse.

In [29]:
# Q7. Création et export du modèle en étoile.
from relational import check_referential_integrity

star_dir = PROCESSED_DIR / 'star'
star_dir.mkdir(parents=True, exist_ok=True)

dim_customer = customers.drop_duplicates('id_client').copy()
dim_product = products.drop_duplicates('id_produit').copy()
dim_store = stores.drop_duplicates('id_magasin').copy()
dim_date = dim_date.drop_duplicates('cle_date').copy()

# Chaque clé de vente sans dimension reçoit une ligne descriptive « inconnue ».
# Cela conserve les faits et rend la clé étrangère valide sans inventer d'attributs.
def add_unknown_dimension_rows(dimension, key, fact_keys, label_column=None):
    known_keys = set(dimension[key].dropna())
    missing_keys = pd.Index(fact_keys.dropna().unique()).difference(known_keys)
    if len(missing_keys):
        placeholders = []
        for missing_key in missing_keys:
            row = {column: pd.NA for column in dimension.columns}
            row[key] = missing_key
            if label_column is not None:
                row[label_column] = f"Inconnu ({missing_key})"
            placeholders.append(row)
        dimension = pd.concat(
            [dimension, pd.DataFrame(placeholders)], ignore_index=True
        )
    return dimension

dim_customer = add_unknown_dimension_rows(
    dim_customer, 'id_client', sales['id_client']
)
dim_product = add_unknown_dimension_rows(
    dim_product, 'id_produit', sales['id_produit'], label_column='libelle'
)
dim_store = add_unknown_dimension_rows(
    dim_store, 'id_magasin', sales['id_magasin'], label_column='nom_magasin'
)

fact_sales = sales.assign(
    cle_date=sales['date_commande'].dt.strftime('%Y%m%d').astype('int64'),
    montant_net=sales['montant'] * (1 - sales['remise_pct'].fillna(0) / 100),
)[[
    'cle_date', 'id_client', 'id_produit', 'id_magasin',
    'quantite', 'prix_unitaire', 'remise_pct', 'montant',
    'montant_net', 'statut',
]].copy()

star_dimensions = {
    'cle_date': dim_date,
    'id_client': dim_customer,
    'id_produit': dim_product,
    'id_magasin': dim_store,
}
check_referential_integrity(fact_sales, star_dimensions)

star_tables = {
    'fact_sales': fact_sales,
    'dim_date': dim_date,
    'dim_customer': dim_customer,
    'dim_product': dim_product,
    'dim_store': dim_store,
}
for table_name, table in star_tables.items():
    output_path = star_dir / f'{table_name}.parquet'
    table.to_parquet(output_path, index=False)
    print(f"{table_name:<14} {table.shape} -> {output_path.relative_to(ROOT)}")

cle_date : 0 clé(s) orpheline(s)
id_client : 0 clé(s) orpheline(s)
id_produit : 0 clé(s) orpheline(s)
id_magasin : 0 clé(s) orpheline(s)
fact_sales     (16992, 10) -> data\processed\star\fact_sales.parquet
dim_date       (730, 10) -> data\processed\star\dim_date.parquet
dim_customer   (3060, 5) -> data\processed\star\dim_customer.parquet
dim_product    (40, 5) -> data\processed\star\dim_product.parquet
dim_store      (9, 5) -> data\processed\star\dim_store.parquet


In [30]:
# Q8. Le contrôle est dans src/relational.py : il affiche chaque compte
# et lève ValueError dès qu'une clé étrangère n'existe pas dans sa dimension.
from relational import check_referential_integrity

orphan_counts = check_referential_integrity(
    fact_sales,
    {
        'cle_date': dim_date,
        'id_client': dim_customer,
        'id_produit': dim_product,
        'id_magasin': dim_store,
    },
)
print('Intégrité référentielle vérifiée : aucune clé orpheline.')

cle_date : 0 clé(s) orpheline(s)
id_client : 0 clé(s) orpheline(s)
id_produit : 0 clé(s) orpheline(s)
id_magasin : 0 clé(s) orpheline(s)
Intégrité référentielle vérifiée : aucune clé orpheline.


### Schéma du modèle en étoile livré

```mermaid
erDiagram
    DIM_DATE ||--o{ FACT_SALES : cle_date
    DIM_CUSTOMER ||--o{ FACT_SALES : id_client
    DIM_PRODUCT ||--o{ FACT_SALES : id_produit
    DIM_STORE ||--o{ FACT_SALES : id_magasin
    DIM_DATE {
        int cle_date PK
        date date
        int annee
        int mois
    }
    DIM_CUSTOMER {
        string id_client PK
        string segment
        string ville
    }
    DIM_PRODUCT {
        string id_produit PK
        string libelle
        string categorie
    }
    DIM_STORE {
        string id_magasin PK
        string nom_magasin
        string region
    }
    FACT_SALES {
        int cle_date FK
        string id_client FK
        string id_produit FK
        string id_magasin FK
        int quantite
        float montant
        float montant_net
    }
```

Les clés étrangères de `fact_sales` pointent vers les clés primaires des quatre dimensions.
L'ID produit `P9999`, absent du référentiel, reçoit une ligne « Inconnu (P9999) » dans
`dim_product` : les 75 ventes sont conservées et l'intégrité référentielle reste vérifiable.

---
## Livrable de la séance

- `data/processed/star/` contenant la table de faits et les quatre dimensions en Parquet ;
- les fonctions `safe_merge` et `check_referential_integrity`, dans `src/` ;
- un schéma (même dessiné à la main et photographié) du modèle en étoile obtenu ;
- le compte des lignes à chaque étape de l'assemblage, avec explication des écarts.

**Ce livrable est le cœur de votre projet.** C'est le fichier que vous ouvrirez dans
Power BI ou Tableau, et celui dont la structure sera discutée en soutenance.